# Notebook 06: Advanced Modeling (XGBoost / LightGBM)

Trains gradient boosting models on engineered features and compares against baselines.

In [ ]:
import pandas as pd
from src.config.config import DATA_PROCESSED_DIR, PRIMARY_TARGET
from src.models.ml_models import train_xgboost, train_lightgbm, feature_importance_df
from src.evaluation.metrics import all_metrics, evaluate_by_segment
from src.visualization.plots import plot_forecast_vs_actual, plot_error_by_retailer


In [ ]:
df = pd.read_parquet(DATA_PROCESSED_DIR / 'features.parquet')
# Define feature columns (all engineered columns)
exclude = ['date', 'units_sold', 'revenue', 'margin']
feature_cols = [c for c in df.columns if c not in exclude]
cat_cols = ['retailer_id', 'object_id', 'category', 'brand', 'region']
print('Features:', len(feature_cols))

In [ ]:
xgb_model, xgb_test, encoders = train_xgboost(
    df, feature_cols, PRIMARY_TARGET, 'date', cat_cols
)
xgb_metrics = all_metrics(xgb_test[PRIMARY_TARGET].values, xgb_test['prediction'].values)
print('XGBoost:', xgb_metrics)

In [ ]:
feature_importance_df(xgb_model, feature_cols).head(20)

In [ ]:
seg_metrics = evaluate_by_segment(xgb_test, PRIMARY_TARGET, 'prediction', ['retailer_id'])
seg_metrics

In [ ]:
plot_forecast_vs_actual(xgb_test).show()
plot_error_by_retailer(seg_metrics).show()